In [1]:
# 1. Force-align Torch with Colab's CUDA 12.x to prevent the 'RuntimeError'
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 2. Web & Tunneling Tools
!pip install fastapi uvicorn pyngrok nest-asyncio pydantic

# 3. Model Loading & Performance (Accelerate is required for 'device_map="auto"')
!pip install transformers accelerate

# 4. ASR & Audio Processing
!pip install openai-whisper librosa

Looking in indexes: https://download.pytorch.org/whl/cu124


In [2]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import librosa
import io
import os
import whisper

device = "cuda" if torch.cuda.is_available() else "cpu"

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

In [3]:
ASRka = whisper.load_model("turbo", device=device)

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    audio_content = await file.read()
    temp_file = "temp_voice.webm"

    with open(temp_file, "wb") as f:
        f.write(audio_content)

    try:
        result = ASRka.transcribe(temp_file, fp16=(device == "cuda"))
        return {"text": result["text"].strip()}
    except Exception as e:
        print(f"ASR Error: {e}")
        return {"text": "Error transcribing audio"}
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)

In [4]:
from pydantic import BaseModel

class Question(BaseModel):
    question: str

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
LLMka = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # Use float16 for speed
    device_map="auto"
)

@app.post("/ask")
async def ask(q: Question):
    # TinyLlama uses a very simple ChatML-style format
    prompt = f"<|system|>\nYou are a helpful assistant.</s>\n<|user|>\n{q.question}</s>\n<|assistant|>\n"

    # 1. Standard tokenization (No special dictionary objects to cause 'shape' errors)
    inputs = tokenizer(prompt, return_tensors="pt").to(LLMka.device)

    # 2. Generate
    with torch.no_grad():
        outputs = LLMka.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150, # Keep it short for speed
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    # 3. Decode only the new part
    prompt_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    return {"answer": response.strip()}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import asyncio

SHOULD_IT_BE_HIDDEN = "..."

ngrok.set_auth_token(SHOULD_IT_BE_HIDDEN)

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("PUBLIC URL:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [20065]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


PUBLIC URL: NgrokTunnel: "https://unheuristically-unanecdotal-simonne.ngrok-free.dev" -> "http://localhost:8000"
INFO:     185.48.148.186:0 - "OPTIONS /ask HTTP/1.1" 200 OK


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     185.48.148.186:0 - "POST /ask HTTP/1.1" 200 OK
INFO:     185.48.148.186:0 - "POST /transcribe HTTP/1.1" 200 OK
INFO:     185.48.148.186:0 - "OPTIONS /ask HTTP/1.1" 200 OK


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     185.48.148.186:0 - "POST /ask HTTP/1.1" 200 OK
INFO:     185.48.148.186:0 - "OPTIONS /ask HTTP/1.1" 200 OK


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     185.48.148.186:0 - "POST /ask HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [20065]
